# PetManager
## Presentación del proyecto final

**Bases de Datos I · Grupo G1 · 2026-2**  
Universidad Industrial de Santander · Ingeniería de Sistemas

| Nombre completo | Código |
|---|---:|
| Jannyer Stiven Caballero Domínguez | 2250191 |
| Carlos Iván Merlano Vergara | 2250188 |
| Andrés Felipe Rivera Carreño | 2250193 |
| Juan Sebastián Araujo Contreras | 2250142 |
| Juan Pablo Vera Suárez | 2241807 |

**Recorrido:** 1. Situación (2 min) · 2. Modelo E-R (2 min) · 3. Modelo relacional (3 min) · 4. SQL (2 min) · 5. Datos (1 min) · 6. Consultas (4 min) · Cierre (1 min).

Este notebook reúne los seis puntos de la entrega. Sus resultados guardados corresponden a una demostración ejecutada en SQLite. Los scripts de implementación están escritos para MySQL 8.0.

## 1. Análisis de la situación

Una veterinaria maneja información de clientes, mascotas, atenciones, personal, insumos y facturas. Cuando estos datos quedan separados, consultar el historial de un animal exige revisar varios registros y se puede repetir información del propietario.

**Pregunta del proyecto:** ¿cómo organizar estos datos en una base relacional que permita conservar las referencias entre la atención de las mascotas y la información administrativa de la veterinaria?

**Objetivo:** diseñar e implementar una base de datos que reúna la información de la veterinaria y permita consultarla mediante SQL.

| Necesidad | Respuesta del modelo |
| --- | --- |
| Identificar al responsable de una mascota | `animal.id_cliente` enlaza cada mascota con su cliente. |
| Consultar antecedentes de atención | Un historial agrupa los registros clínicos del animal. |
| Distinguir los tipos de atención | Cita, vacunación, tratamiento y hospitalización se representan como subtipos. |
| Organizar la operación | Se relacionan veterinarias, sucursales, empleados e inventarios. |
| Consultar cobros e insumos | Las facturas se asocian a clientes; los medicamentos y elementos médicos, a inventarios. |

La entrega implementa estas funciones con 18 tablas. La exploración inicial también consideró pagos, exámenes, lotes, recordatorios, usuarios y permisos, pero esas funciones no forman parte del esquema construido. Tampoco se asigna un veterinario a cada cita ni se relaciona cada factura con una atención específica.

## 2. Modelo entidad-relación

El modelo se organiza en tres grupos: administración de la veterinaria, clientes y atención clínica, e inventario. El diagrama de la primera entrega permite seguir esas relaciones y las dos especializaciones.

| Relación | Interpretación en la implementación |
| --- | --- |
| Veterinaria con sucursales, empleados y clientes | Cada registro hijo pertenece a una veterinaria; una veterinaria puede tener varios. |
| Cliente con animales y facturas | Un cliente puede tener varias mascotas y facturas; cada una pertenece a un cliente. |
| Animal con historial clínico | Cada historial pertenece a un animal y la FK única evita dos historiales para el mismo animal. |
| Historial con registros clínicos | Un historial puede reunir múltiples atenciones en distintas fechas. |
| Sucursal con inventario | Cada inventario pertenece a una sucursal; la FK única impide duplicarlo para esa sucursal. |
| Inventario con medicamentos y elementos médicos | Cada producto registrado pertenece a un inventario. |
| Empleado con veterinario, secretario y auxiliar | Los subtipos comparten el identificador de empleado. |
| Registro clínico con cita, vacunación, tratamiento y hospitalización | Los subtipos comparten el identificador del registro clínico. |

La notación 1:1 del diseño se implementa con una FK única en la tabla hija: asegura como máximo un hijo por padre, pero no obliga a que todos los padres tengan uno. Las especializaciones tampoco imponen exclusividad ni cobertura total mediante PK y FK.


![Modelo entidad-relación de PetManager](../Modelo%20E-R/Modelo/Modelo_Entidad_Relacion.png)

[Ver diagrama en tamaño completo](../Modelo%20E-R/Modelo/Modelo_Entidad_Relacion.png)

## 3. Modelo relacional y normalización

Cada entidad se transforma en una tabla. En las relaciones 1:N, la FK se ubica en el lado N; en las relaciones 1:1, esa FK además es única. Los subtipos utilizan una misma columna como PK y FK.

| Restricción | Ejemplo | Función |
| --- | --- | --- |
| PK | `cliente.id_cliente` | Identifica al cliente. |
| FK | `animal.id_cliente` | Evita una mascota asociada a un cliente inexistente. |
| UK | `factura.numero_factura` | Impide repetir el número de factura. |
| UK sobre FK | `historial_clinico.id_animal` | Limita a un historial por animal. |
| NN | `animal.nombre_animal` | Exige un valor para el atributo. |

### Proceso de normalización

| Etapa | Revisión aplicada |
| --- | --- |
| 1FN | Cada atributo contiene un valor de su dominio; no se almacenan grupos repetidos. |
| 2FN | Las claves candidatas declaradas son simples, por lo que no existen dependencias parciales respecto a ellas. |
| 3FN y BCNF | Bajo las dependencias documentadas, los determinantes son claves y no se introducen dependencias transitivas entre atributos no clave. |
| 4FN y 5FN | Se conserva el modelo bajo el supuesto de que no hay dependencias multivaluadas independientes ni dependencias de reunión adicionales a las declaradas. |
| 6FN | El ejercicio descompone atributos en 54 tablas: 18 tablas base y 36 proyecciones adicionales. |

Por ejemplo, `id_cliente` determina nombre, apellido, teléfono, dirección, correo e identificación de la veterinaria. Los datos de contacto se guardan en `cliente`, por lo que cada mascota solo necesita `id_cliente` para relacionarse con su propietario.

Para SQL se conservan las **18 tablas de la propuesta de implementación**, lo que evita las uniones adicionales de la descomposición en 6FN. Esta decisión coincide con la recomendación del informe. La afirmación de normalización depende de las reglas de negocio declaradas; no se deduce únicamente de tener claves primarias.

En 6FN, los atributos obligatorios requieren cobertura de los identificadores de su tabla base. La observación clínica es opcional y se reconstruye con `LEFT JOIN`. En las 18 tablas implementadas esa condición se expresa como `observacion TEXT NULL`.


[Consultar Excel del modelo](../Modelo%20Relacional/Modelo%20Relacional%20Normalizado/ModeloRelacionalNormalizadoPetManager.xlsx) · [Leer informe de normalización](../Modelo%20Relacional/Informe%20de%20Normalizaci%C3%B3n/Informe_Normalizacion_PetManager.pdf)

### Tablas del modelo relacional

| Tabla | PK | FK |
| --- | --- | --- |
| veterinaria | id_veterinaria | — |
| sucursal | id_sucursal | id_veterinaria → veterinaria.id_veterinaria |
| inventario | id_inventario | id_sucursal → sucursal.id_sucursal |
| elemento_medico | id_elemento | id_inventario → inventario.id_inventario |
| medicamento | id_medicamento | id_inventario → inventario.id_inventario |
| empleado | id_empleado | id_veterinaria → veterinaria.id_veterinaria |
| veterinario | id_empleado | id_empleado → empleado.id_empleado |
| secretario | id_empleado | id_empleado → empleado.id_empleado |
| auxiliar | id_empleado | id_empleado → empleado.id_empleado |
| cliente | id_cliente | id_veterinaria → veterinaria.id_veterinaria |
| factura | id_factura | id_cliente → cliente.id_cliente |
| animal | id_animal | id_cliente → cliente.id_cliente |
| historial_clinico | id_historial | id_animal → animal.id_animal |
| registro_clinico | id_registro | id_historial → historial_clinico.id_historial |
| cita | id_registro | id_registro → registro_clinico.id_registro |
| vacunacion | id_registro | id_registro → registro_clinico.id_registro |
| tratamiento | id_registro | id_registro → registro_clinico.id_registro |
| hospitalizacion | id_registro | id_registro → registro_clinico.id_registro |

## 4. Base de datos construida en SQL

El esquema contiene 18 tablas y 17 claves foráneas. Se crean primero las tablas padre para que las referencias existan cuando se crean las tablas hijas. Los identificadores propios usan `AUTO_INCREMENT`; los subtipos heredan la PK de su tabla principal.

Se utiliza `DECIMAL` para salarios y montos. `cita.hora` es `TIME`, mientras que la fecha de atención se obtiene de `registro_clinico.fecha_registro`. El inventario conserva `numero_stock` como dato cargado; `cantidad_total` es un atributo derivado que no se almacena en una columna.

La creación nativa se realiza con los scripts de MySQL. La demostración ejecutable del notebook utiliza SQLite en memoria: elimina `USE`, `AUTO_INCREMENT` y `ENGINE=InnoDB`, y cambia `START TRANSACTION` por `BEGIN TRANSACTION`. Los identificadores ya están presentes en los datos y se activan las FK. Las diez consultas conservan sus sentencias SQL.

Esta demostración comprueba los datos y sus relaciones, pero no valida el comportamiento específico de los tipos, la intercalación ni el autoincremento de MySQL.


[Guía de ejecución nativa en MySQL](../SQL/README.md) · [Esquema completo](../SQL/02_Creacion_de_tablas/basededatosVeterinaria.sql)

Ejemplo de una relación 1:1 y de una especialización:

```sql
CREATE TABLE historial_clinico (
    id_historial INT NOT NULL AUTO_INCREMENT,
    id_animal INT NOT NULL,
    numero_registro INT NOT NULL,
    fecha_apertura DATE NOT NULL,
    UNIQUE(id_animal),
    PRIMARY KEY (id_historial),
    FOREIGN KEY (id_animal) REFERENCES animal(id_animal)
) ENGINE=InnoDB;

CREATE TABLE vacunacion (
    id_registro INT NOT NULL,
    tipo_vacuna VARCHAR(50) NOT NULL,
    dosis VARCHAR(60) NOT NULL,
    PRIMARY KEY (id_registro),
    FOREIGN KEY (id_registro) REFERENCES registro_clinico(id_registro)
) ENGINE=InnoDB;
```

### Preparar la demostración

Ejecuta las celdas en orden con Python 3. El notebook localiza la raíz del repositorio desde su propia carpeta o desde la raíz. Necesita que se haya descargado el repositorio completo. La base se crea de nuevo en memoria cada vez que ejecutas esta celda.

In [1]:
from pathlib import Path
import sys
import sqlite3
import importlib.util
from html import escape
from IPython.display import display, HTML, Markdown

actual = Path.cwd().resolve()
raiz = next((p for p in [actual, *actual.parents]
             if (p / "SQL" / "validar_proyecto.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Descarga el repositorio completo y abre el notebook dentro de Petmanager.")
spec = importlib.util.spec_from_file_location("petmanager_sql", raiz / "SQL" / "validar_proyecto.py")
proyecto = importlib.util.module_from_spec(spec)
spec.loader.exec_module(proyecto)
if "conexion" in globals():
    conexion.close()
conexion = proyecto.crear_demo()

def mostrar_tabla(columnas, filas):
    cabecera = "".join("<th>" + escape(str(c)) + "</th>" for c in columnas)
    cuerpo = "".join("<tr>" + "".join("<td>" + escape(str(v)) + "</td>" for v in fila) + "</tr>" for fila in filas)
    display(HTML("<table><thead><tr>" + cabecera + "</tr></thead><tbody>" + cuerpo + "</tbody></table>"))

print("Motor de esta demostración: SQLite", sqlite3.sqlite_version)
print("Base en memoria creada con las 18 tablas y los datos de los scripts.")

Motor de esta demostración: SQLite 3.53.1
Base en memoria creada con las 18 tablas y los datos de los scripts.


## 5. Carga de datos al modelo

El archivo de carga contiene 230 registros. Las entidades principales se insertan antes de sus tablas dependientes y cada `INSERT` indica explícitamente sus columnas.

Hay 30 empleados, 40 registros clínicos y 10 registros en cada una de las otras 16 tablas. Los 40 registros clínicos corresponden a diez citas, diez vacunaciones, diez tratamientos y diez hospitalizaciones. Se cumple el requisito de al menos cinco registros por tabla.

In [2]:
conteos = proyecto.conteos(conexion)
mostrar_tabla(["Tabla", "Registros"], conteos)
print("Total:", sum(n for _, n in conteos))
print("Todas las tablas cumplen el mínimo de 5:", all(n >= 5 for _, n in conteos))

Tabla,Registros
animal,10
auxiliar,10
cita,10
cliente,10
elemento_medico,10
empleado,30
factura,10
historial_clinico,10
hospitalizacion,10
inventario,10


Total: 230
Todas las tablas cumplen el mínimo de 5: True


### Comprobar relaciones y resultados

La siguiente celda verifica los conteos, las claves y las diez consultas. Los intentos de insertar filas inválidas se revierten y no alteran los datos de la demostración.

In [3]:
proyecto.comprobar(conexion)

Motor de la demostración: SQLite 3.53.1
18 tablas, 17 claves foráneas y 230 registros; mínimo por tabla: 10.
Sin referencias huérfanas. PK, tres UK y observación opcional comprobadas.
Filas por consulta: 3, 2, 5, 10, 28, 6, 10, 10, 3, 3
Las diez consultas devuelven los resultados esperados.
La ejecución nativa en MySQL se realiza con los scripts de SQL/README.md.


## 6. Consultas sobre la base de datos

Las diez consultas responden preguntas sobre clientes, salarios, medicamentos, inventarios y atención clínica. Se utilizan `WHERE`, `BETWEEN`, `IN`, `NOT IN`, `LIKE`, `ORDER BY` y subconsultas con `EXISTS`.

`EXISTS` permite saber si hay al menos una fila relacionada sin duplicar al cliente o a la mascota cuando tiene varios registros que cumplen la condición. En las consultas 7 y 10 se recorre la relación animal, historial, registro clínico y tipo de atención.


| N.º | Consulta | Filas |
| --- | --- | --- |
| 1 | Clientes con mascotas de determinadas razas | 3 |
| 2 | Medicamentos con pocas existencias | 2 |
| 3 | Clientes con facturas de valores elevados | 5 |
| 4 | Historiales abiertos en el primer trimestre de 2026 | 10 |
| 5 | Empleados según su salario | 28 |
| 6 | Auxiliares de otras áreas | 6 |
| 7 | Mascotas con vacunación registrada | 10 |
| 8 | Tratamientos según sus indicaciones | 10 |
| 9 | Inventarios que requieren revisión | 3 |
| 10 | Mascotas con hospitalizaciones prolongadas | 3 |

In [4]:
consultas = proyecto.consultas()
def ejecutar_consulta(numero):
    cursor = conexion.execute(consultas[numero - 1])
    filas = cursor.fetchall()
    mostrar_tabla([col[0] for col in cursor.description], filas)
    print("Filas obtenidas:", len(filas))

### Consulta 1. Clientes con mascotas de determinadas razas

Localizar propietarios de mascotas Labrador, Golden Retriever o Beagle.

```sql
SELECT nombre_cli, apellido_cli, telefono_cli, correo
FROM cliente
WHERE id_cliente IN (
    SELECT id_cliente
    FROM animal
    WHERE raza IN ('Labrador', 'Golden Retriever', 'Beagle')
)
ORDER BY id_cliente;
```

In [5]:
ejecutar_consulta(1)

nombre_cli,apellido_cli,telefono_cli,correo
Andres,Gomez,3201111111,andres@gmail.com
Juan,Lopez,3205555555,juan@gmail.com
Pedro,Castro,3209999999,pedro@gmail.com


Filas obtenidas: 3


**Lectura del resultado:** Andres Gomez, Juan Lopez y Pedro Castro.

### Consulta 2. Medicamentos con pocas existencias

Revisar antibióticos, antiinflamatorios y analgésicos con entre 1 y 30 unidades.

```sql
SELECT tipo_medicamento, presentacion, dosis, cantidad
FROM medicamento
WHERE cantidad BETWEEN 1 AND 30
AND tipo_medicamento IN (
    'Antibiótico',
    'Antiinflamatorio',
    'Analgésico'
)
ORDER BY cantidad ASC;
```

In [6]:
ejecutar_consulta(2)

tipo_medicamento,presentacion,dosis,cantidad
Antiinflamatorio,Inyectable,20 mg/ml,20
Antibiótico,Inyectable,1 g,25


Filas obtenidas: 2


**Lectura del resultado:** Antiinflamatorio inyectable: 20 unidades; antibiótico inyectable: 25. El filtro no incluye productos agotados.

### Consulta 3. Clientes con facturas de valores elevados

Identificar clientes con alguna factura entre $100.001 y $200.000.

```sql
SELECT nombre_cli, apellido_cli, telefono_cli, correo
FROM cliente c
WHERE EXISTS (
    SELECT 1
    FROM factura f
    WHERE f.id_cliente = c.id_cliente
    AND f.monto BETWEEN 100001 AND 200000
)
ORDER BY c.id_cliente;
```

In [7]:
ejecutar_consulta(3)

nombre_cli,apellido_cli,telefono_cli,correo
Laura,Martinez,3202222222,laura@gmail.com
Maria,Rodriguez,3204444444,maria@gmail.com
Daniel,Torres,3207777777,daniel@gmail.com
Camila,Diaz,3208888888,camila@gmail.com
Pedro,Castro,3209999999,pedro@gmail.com


Filas obtenidas: 5


**Lectura del resultado:** Laura Martinez, Maria Rodriguez, Daniel Torres, Camila Diaz y Pedro Castro.

### Consulta 4. Historiales abiertos en el primer trimestre de 2026

Consultar la apertura de historiales entre el 1 de enero y el 31 de marzo.

```sql
SELECT id_animal, numero_registro,  fecha_apertura
FROM historial_clinico
WHERE fecha_apertura BETWEEN '2026-01-01' AND '2026-03-31'
ORDER BY fecha_apertura DESC;
```

In [8]:
ejecutar_consulta(4)

id_animal,numero_registro,fecha_apertura
10,5010,2026-02-05
9,5009,2026-02-01
8,5008,2026-01-28
7,5007,2026-01-25
6,5006,2026-01-22
5,5005,2026-01-20
4,5004,2026-01-18
3,5003,2026-01-15
2,5002,2026-01-12
1,5001,2026-01-10


Filas obtenidas: 10


**Lectura del resultado:** Los diez historiales están dentro del periodo consultado.

### Consulta 5. Empleados según su salario

Filtrar salarios entre $1.500.000 y $2.000.000 o superiores a $3.500.000.

```sql
SELECT id_empleado, nombre_emp, apellido_emp, telefono_emp, salario
FROM empleado
WHERE salario BETWEEN 1500000 AND 2000000
OR salario > 3500000
ORDER BY salario DESC;
```

In [9]:
ejecutar_consulta(5)

id_empleado,nombre_emp,apellido_emp,telefono_emp,salario
8,Valentina,Diaz,3108888888,4100000
4,Camila,Perez,3104444444,4000000
10,Maria,Herrera,3110000000,4000000
6,Sofia,Ramirez,3106666666,3900000
2,Laura,Gomez,3102222222,3800000
9,Sebastian,Lopez,3109999999,3800000
5,Daniel,Torres,3105555555,3700000
3,Andres,Rodriguez,3103333333,3600000
13,Luisa,Fernandez,3113333333,1900000
17,Diana,Ortiz,3117777777,1900000


Filas obtenidas: 28


**Lectura del resultado:** 28 empleados. Los empleados 1 y 7 quedan fuera porque ganan exactamente $3.500.000.

### Consulta 6. Auxiliares de otras áreas

Consultar auxiliares de áreas distintas de Laboratorio y Farmacia.

```sql
SELECT id_empleado, area_apoyo
FROM auxiliar
WHERE area_apoyo NOT IN ('Laboratorio', 'Farmacia')
ORDER BY area_apoyo ASC;
```

In [10]:
ejecutar_consulta(6)

id_empleado,area_apoyo
22,Cirugia
27,Cirugia
23,Hospitalizacion
28,Hospitalizacion
24,Urgencias
29,Urgencias


Filas obtenidas: 6


**Lectura del resultado:** Seis auxiliares: identificadores 22, 23, 24, 27, 28 y 29.

### Consulta 7. Mascotas con vacunación registrada

Recorrer mascota, historial y registro clínico para comprobar la existencia de una vacunación.

```sql
SELECT id_animal, nombre_animal, raza, sexo
FROM animal a
WHERE EXISTS (
    SELECT 1
    FROM historial_clinico h
    WHERE h.id_animal = a.id_animal
    AND EXISTS (
        SELECT 1
        FROM registro_clinico r
        WHERE r.id_historial = h.id_historial
        AND EXISTS (
            SELECT 1
            FROM vacunacion v
            WHERE v.id_registro = r.id_registro
        )
    )
)
ORDER BY a.id_animal;
```

In [11]:
ejecutar_consulta(7)

id_animal,nombre_animal,raza,sexo
1,Max,Labrador,Macho
2,Luna,Persa,Hembra
3,Rocky,Bulldog,Macho
4,Mia,Siamés,Hembra
5,Toby,Golden Retriever,Macho
6,Nala,Criollo,Hembra
7,Bruno,Pastor Aleman,Macho
8,Coco,Poodle,Macho
9,Lola,Beagle,Hembra
10,Simba,Mestizo,Macho


Filas obtenidas: 10


**Lectura del resultado:** Las diez mascotas tienen una vacunación registrada.

### Consulta 8. Tratamientos según sus indicaciones

Buscar indicaciones que contengan Administrar o Aplicar.

```sql
SELECT id_registro, receta_medica, indicaciones
FROM tratamiento
WHERE indicaciones LIKE '%Administrar%'
OR indicaciones LIKE '%Aplicar%'
ORDER BY receta_medica ASC;
```

In [12]:
ejecutar_consulta(8)

id_registro,receta_medica,indicaciones
27,Analgesico,Administrar cada 24 horas
21,Antibiotico oral,Administrar cada 12 horas
26,Antihistaminico,Administrar cada 12 horas
24,Antiinflamatorio,Administrar cada 24 horas
25,Antiparasitario,Administrar dosis indicada
30,Complejo vitaminico,Administrar diariamente
22,Crema dermatologica,Aplicar dos veces al dia
29,Desparasitante,Administrar dosis unica
28,Gotas oftalmicas,Aplicar tres veces al dia
23,Protector gastrico,Administrar antes de comer


Filas obtenidas: 10


**Lectura del resultado:** Los diez tratamientos cumplen el filtro.

### Consulta 9. Inventarios que requieren revisión

Filtrar el valor almacenado en numero_stock cuando es menor que 100 o mayor que 200.

```sql
SELECT id_inventario, id_sucursal, numero_stock
FROM inventario
WHERE numero_stock < 100
OR numero_stock > 200
ORDER BY numero_stock DESC;
```

In [13]:
ejecutar_consulta(9)

id_inventario,id_sucursal,numero_stock
7,7,250
10,10,220
6,6,90


Filas obtenidas: 3


**Lectura del resultado:** Inventarios 7 (250), 10 (220) y 6 (90). Los valores 100 y 200 no se incluyen.

### Consulta 10. Mascotas con hospitalizaciones prolongadas

Encontrar mascotas con una hospitalización de más de tres días.

```sql
SELECT id_animal, nombre_animal, raza, sexo
FROM animal a
WHERE EXISTS (
    SELECT 1
    FROM historial_clinico h
    WHERE h.id_animal = a.id_animal
    AND EXISTS (
        SELECT 1
        FROM registro_clinico r
        WHERE r.id_historial = h.id_historial
        AND EXISTS (
            SELECT 1
            FROM hospitalizacion hp
            WHERE hp.id_registro = r.id_registro
            AND hp.duracion > 3
        )
    )
)
ORDER BY a.id_animal;
```

In [14]:
ejecutar_consulta(10)

id_animal,nombre_animal,raza,sexo
3,Rocky,Bulldog,Macho
6,Nala,Criollo,Hembra
9,Lola,Beagle,Hembra


Filas obtenidas: 3


**Lectura del resultado:** Rocky (4 días), Nala (5 días) y Lola (4 días).

## Cierre

PetManager reúne la información de la veterinaria en 18 tablas relacionadas. La carga inicial cumple el mínimo de registros y las diez consultas permiten revisar clientes, inventarios y atenciones. Las claves separan la información de cada entidad y permiten recuperar los datos mediante sus relaciones.

**Material de apoyo:** [scripts SQL](../SQL/README.md), [ajustes del esquema](../SQL/03_Modificaciones/README.md) y [guía de sustentación](./README.md).